# Day 2

Today, we will start using nf-core pipelines to find differentially abundant genes in our dataset. 
We are using data from the following paper: https://www.nature.com/articles/s41593-023-01350-3#Sec10

1. Please take some time to read through the paper and understand their approach, hypotheses and goals.

What was the objective of the study?

The objective of the study was to investigate how oxycodone withdrawl affects the gene expression adaptions in the nucleus accumbens, medial prefrontal cortex and vetral tegmental area, to better understand and treat the effects of opioid withdrawl on patients with chronic pain. 
The aim was to potentially identify if inhibition of HDAC1/HDAC2 provides an approach for the treatment of patient with chronic pain dependent on opioids. 
Hypothesiszed that certain areas are differentially expressed in mice with oxycodon and chronic pain. 

What do the conditions mean?

oxy: mice receiving (chronic) Oxycodon injections


sal: mice receiving saline injections

What do the genotypes mean?

SNI: surgery to induce (prolonged) spared nerve injury


Sham: Sham or placebo surgery

Imagine you are the bioinformatician in the group who conducted this study. They hand you the raw files and ask you to analyze them.

1. What would you do?

2. Which groups would you compare to each other?

Please also mention which outcome you would expect to see from each comparison.

1. First I would pre-process the data by normalising it and by performing exploratory statistical measures(such as a pca, heatmap, etc.), to ensure that the data is "usable" (Quality control)→ E.g. an acceptable amount of outliers, a reasonable/expected distribution and so on. I'd also check all genes have the labeling that is necessary, and might pull missing labels from databases. Than I would move on to an differential analysis, where i am comparing the gene expression profiles of the different groups. Differential abundance analysis. 

2. First of all, as the aim of the study was to investigate oxicodon withdrawl in sham and SNI mice, I'd take a close look at the differences in the expression profile in these mice. As it is also important to distinguish if these changes are happening just due to the chronic pain, it would be good to compare this expression profile to the one compared between the sal-sham and sal-sni mice. I would expect to see expression changes between the sal-sni and sal-sham groups within the vetral tegmental area, medial prefrontal cortex, and nucleus accumbens. Additionally I would expect differential expression associated with oxicodon withdrawl between the sal(sham+sni) and oxy(sham+sni) groups. I would than expect a unique set of expressions specifically for the sni-oxy group compared to the sham-oxy group that would indicate the heightened pain sensitivity(and other symptoms) of the sni-oxy group. 

Oxicodon to saline group, than check if sham has different behaviour, to rule out 

Your group gave you a very suboptimal excel sheet (conditions_runs_oxy_project.xlsx) to get the information you need for each run they uploaded to the SRA.<br>
So, instead of directly diving into downloading the data and starting the analysis, you first need to sort the lazy table.<br>
Use Python and Pandas to get the table into a more sensible order.<br>
Then, perform some overview analysis and plot the results
1. How many samples do you have per condition?
2. How many samples do you have per genotype?
3. How often do you have each condition per genotype?

In [30]:
import pandas as pd
# read file into pandas dataframe 
oxy_df = pd.read_excel("/home/nina/computational-workflows-2026/day_02/conditions_runs_oxy_project.xlsx")
oxy_df["Conditions"] = oxy_df["condition: Sal"].notna().map({True: "Sal", False: "Oxy"})
oxy_df["Genotype"] = oxy_df["Genotype: SNI"].notna().map({True: "Sham", False: "SNI"})
oxy_df = oxy_df.drop(columns=["condition: Sal","Condition: Oxy","Genotype: Sham", "Genotype: SNI","Patient", "RNA-seq", "DNA-seq"])

print(oxy_df)

            Run Conditions Genotype
0   SRR23195505        Sal     Sham
1   SRR23195506        Oxy      SNI
2   SRR23195507        Sal      SNI
3   SRR23195508        Oxy     Sham
4   SRR23195509        Oxy     Sham
5   SRR23195510        Sal     Sham
6   SRR23195511        Oxy      SNI
7   SRR23195512        Sal      SNI
8   SRR23195513        Sal     Sham
9   SRR23195514        Oxy      SNI
10  SRR23195515        Sal      SNI
11  SRR23195516        Oxy     Sham
12  SRR23195517        Oxy     Sham
13  SRR23195518        Sal     Sham
14  SRR23195519        Oxy      SNI
15  SRR23195520        Sal      SNI


In [33]:
# count number of samples per condition
grouped_counts_con = oxy_df.groupby('Conditions')['Conditions'].value_counts()
print(grouped_counts_con)


Conditions
Oxy    8
Sal    8
Name: count, dtype: int64


In [34]:
# count number of samples per genotype
grouped_counts_geno = oxy_df.groupby('Genotype')['Genotype'].value_counts()
print(grouped_counts_geno)

Genotype
SNI     8
Sham    8
Name: count, dtype: int64


In [ ]:
# Each condition per Genotype
oxy_df.groupby(["Genotype", "Conditions"]).size()


Genotype  Conditions
SNI       Oxy           4
          Sal           4
Sham      Oxy           4
          Sal           4
dtype: int64

They were so kind to also provide you with the information of the number of bases per run, so that you can know how much space the data will take on your Cluster.<br>
Add a new column to your fancy table with this information (base_counts.csv) and sort your dataframe according to this information and the condition.

Then select the 2 smallest runs from your dataset and download them from SRA (maybe an nf-core pipeline can help here?...)

In [66]:
base_counts_df = pd.read_csv("/home/nina/computational-workflows-2026/day_02/base_counts.csv")

new_oxy_df = oxy_df.merge(base_counts_df[["Run", "Bases"]],
                          on="Run", how="left").sort_values(by ="Bases")

smallest_run_df = oxy_df.iloc[:2][["Run"]]

smallest_run_df.to_csv('Smalles_runs.csv', header=False, index=False)
smallest_run_df



,Run
0,SRR23195505
1,SRR23195506


In [ ]:
!nextflow run nf-core/fetchngs \
   -profile docker \
   --input Smalles_runs.csv \
   --outdir smallest_runs


 N E X T F L O W   ~  version 26.04.6

Launching `https://github.com/nf-core/fetchngs` [deadly_jang] revision: 955c892d80 [master]

WARN: Unrecognized config option 'manifest.diagram'

------------------------------------------------------
                                        ,--./,-.
        ___     __   __   __   ___     /,-._.--~'
  |\ | |__  __ /  ` /  \ |__) |__         }  {
  | \| |       \__, \__/ |  \ |___     \`-._,-`-,
                                        `._,._,'
  nf-core/fetchngs 1.13.0
------------------------------------------------------

Input/output options
  input                     : Smalles_runs.csv
  outdir                    : smallest_runs

Institutional config options
  config_profile_name       : Test profile
  config_profile_description: Minimal test dataset to check pipeline function

Deprecated options
  trace_report_suffix       : 2026-09-29_12-06-12

Core Nextflow options
  revision                  : master
  runName                   : deadly_ja

While your files are downloading, get back to the paper and explain how you would try to reproduce the analysis.<br>
When you are done with this shout, so we can discuss the different ideas.

Reproducing the exact results would be rather hard, as the authors of the paper dont describe all the steps they took in detail. Especially the data pre-processing and data normalisation steps are not really mentioned. 